In [ ]:
%cd ..

/workspace/code-suggester


In [ ]:
from unsloth import FastLanguageModel
import json
import torch
import re

import json
import ast
import pandas as pd
from pyflakes.api import check
from pyflakes.reporter import Reporter
from io import StringIO

import asyncio
import os
import pandas as pd
from openai import AsyncOpenAI
import nest_asyncio
from tqdm.asyncio import tqdm_asyncio

from configs.env import settings

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/workspace/code-suggester/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0922 11:49:10.675000 11901 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0922 11:49:10.700000 11901 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


🦥 Unsloth Zoo will now patch everything to make training faster!


/workspace/code-suggester/.venv/lib/python3.13/site-packages/unsloth/import_fixes.py:3525: FutureWarning: torch._dynamo.config.inline_inbuilt_nn_modules is deprecated and does not do anything, inline_inbuilt_nn_modules is always True. It will be removed in a future version of PyTorch.
  original_setattr(self, name, value)


In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
LORA_MODEL = "hn-minh/Qwen2.5-Coder-7B-Instruct-backend-LoRA"
DATASET_PATH = "data/processed/eval_dataset.json"

SYSTEM_PROMPT = """You are an elite AI Backend Engineer. Your sole task is to translate natural language requirements into precise, production-grade Python code for AI systems (e.g., FastAPI, SQLAlchemy, Vector databases, LLM integrations).

    Strict Output Rules:
    1. Output ONLY the raw Python code.
    2. Do NOT use markdown code blocks (e.g., do not wrap the output in ```python ... ``` or ```).
    3. Do NOT provide any explanations, context, warnings, or conversational filler (e.g., do not say "Here is the code:" or "Sure!").
    4. Enforce production best practices implicitly: utilize async/await for I/O-bound operations, include strict type hints, ensure proper resource management (e.g., context managers for database sessions), and never block the event loop."""

In [ ]:
def generate_responses(model, tokenizer, dataset, output_file):
    results = []
    for item in dataset:
        prompt = item["messages"][0]["content"]
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]

        inputs = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")

        outputs = model.generate(input_ids=inputs, temperature=0.1)
        generated_text = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)

        results.append({
            "id": item["id"],
            "generated_code": generated_text
        })
        print(f"Generated for {item['id']}")

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print(f"Saved to {output_file}")

In [ ]:
with open(DATASET_PATH, "r") as f:
    eval_data = json.load(f)

print("Loading Base Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True
)
FastLanguageModel.for_inference(model)

print("\n--- INFERENCE: BASE MODEL ---")
generate_responses(model, tokenizer, eval_data, "results_base.json")

print("\n Loading LoRA Adapter...")
model.load_adapter(LORA_MODEL)

print("\n--- INFERENCE: LORA MODEL ---")
generate_responses(model, tokenizer, eval_data, "results_lora.json")

Loading Base Model...
==((====))==  Unsloth 2026.9.7: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.802 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.12.1+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.7.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 339/339 [00:00<00:00, 403.08it/s]
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



--- INFERENCE: BASE MODEL ---
Generated for 1
Generated for 2
Generated for 3
Generated for 4
Generated for 5
Generated for 6
Generated for 7
Generated for 8
Generated for 9
Generated for 10
Generated for 11
Generated for 12
Generated for 13
Generated for 14
Generated for 15
Generated for 16
Generated for 17
Generated for 18
Generated for 19
Generated for 20
Generated for 21
Generated for 22
Generated for 23
Generated for 24
Generated for 25
Generated for 26
Generated for 27
Generated for 28
Generated for 29
Generated for 30
Saved to results_base.json

 Loading LoRA Adapter...


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 392/392 [00:00<00:00, 1772.98it/s]



--- INFERENCE: LORA MODEL ---
Generated for 1
Generated for 2
Generated for 3
Generated for 4
Generated for 5
Generated for 6
Generated for 7
Generated for 8
Generated for 9
Generated for 10
Generated for 11
Generated for 12
Generated for 13
Generated for 14
Generated for 15
Generated for 16
Generated for 17
Generated for 18
Generated for 19
Generated for 20
Generated for 21
Generated for 22
Generated for 23
Generated for 24
Generated for 25
Generated for 26
Generated for 27
Generated for 28
Generated for 29
Generated for 30
Saved to results_lora.json


In [ ]:
def check_executability(code_string):
    if not code_string.strip():
        return False, "Empty code"

    try:
        ast.parse(code_string)
    except SyntaxError as e:
        return False, f"SyntaxError: {e.msg} at line {e.lineno}"
    except Exception as e:
        return False, f"ParseError: {str(e)}"

    error_stream = StringIO()
    warning_stream = StringIO()
    reporter = Reporter(warning_stream, error_stream)

    check(code_string, filename="<llm_code>", reporter=reporter)

    errors = error_stream.getvalue()
    if errors:
        if "undefined name" in errors:
            return False, f"Undefined Variable/Import:\n{errors.strip().splitlines()[0]}"

    return True, "Pass (Executable)"

def evaluate(benchmark_file, base_file, lora_file):
    with open(benchmark_file, 'r') as f: benchmarks = {b['id']: b for b in json.load(f)}
    with open(base_file, 'r') as f: base_results = {b['id']: b['generated_code'] for b in json.load(f)}
    with open(lora_file, 'r') as f: lora_results = {b['id']: b['generated_code'] for b in json.load(f)}

    report = []

    for task_id in benchmarks.keys():
        base_code = base_results.get(task_id, "")
        lora_code = lora_results.get(task_id, "")

        base_pass, base_msg = check_executability(base_code)
        lora_pass, lora_msg = check_executability(lora_code)

        report.append({
            "Task ID": task_id,
            "Base Executable": "✅ Pass" if base_pass else f"❌ {base_msg}",
            "LoRA Executable": "✅ Pass" if lora_pass else f"❌ {lora_msg}",
            "Base Pass": base_pass,
            "LoRA Pass": lora_pass
        })

    df = pd.DataFrame(report)

    base_pass_rate = df['Base Pass'].mean() * 100
    lora_pass_rate = df['LoRA Pass'].mean() * 100

    print("=========================================")
    print(f"🏆 EXECUTABILITY (PASS@1) BENCHMARK RESULTS")
    print("=========================================")
    print(f"Base Model Pass Rate: {base_pass_rate:.1f}%")
    print(f"LoRA Model Pass Rate: {lora_pass_rate:.1f}%")
    print("-----------------------------------------")

    if lora_pass_rate > base_pass_rate:
        print(f"LoRA model improved executability by {lora_pass_rate - base_pass_rate:.1f}%.")
    elif lora_pass_rate < base_pass_rate:
        print(f"LoRA model dropped executability by {base_pass_rate - lora_pass_rate:.1f}%.")
    else:
        print("⚖️ TIE! Both models have the same executability rate.")

    lora_wins = df[(df['Base Pass'] == False) & (df['LoRA Pass'] == True)]
    if not lora_wins.empty:
        print("\n🌟 Tasks that are fixed:")
        for _, row in lora_wins.iterrows():
            print(f" - {row['Task ID']} (Base error: {row['Base Executable'].replace('❌ ', '')})")

    df.drop(columns=['Base Pass', 'LoRA Pass']).to_csv("executability_report.csv", index=False)
    print("\n Saved report")

  evaluate("../data/processed/eval_dataset.json", "results_base.json", "results_lora.json")

In [ ]:
nest_asyncio.apply()

OPENAI_API_KEY = settings.OPENAI_API_KEY

client = AsyncOpenAI(api_key=OPENAI_API_KEY)
SEMAPHORE = asyncio.Semaphore(5)

In [ ]:
async def llm_judge(prompt, generated_code, criteria):
    system_prompt = """You are an expert Senior AI Backend Engineer acting as a strict code reviewer.
Evaluate the provided Python code based ONLY on the given criteria.

Return a valid JSON object exactly in this format:
{
    "score": <int from 1 to 5>,
    "reason": "<short explanation, max 2 sentences>"
}

Score Guide:
5: Flawless, perfectly implements criteria and handles edge cases.
4: Minor issues but mostly correct.
3: Works but misses some edge cases or best practices mentioned in criteria.
2: Major logic flaws, violates criteria.
1: Completely wrong, dangerous, or throws immediate errors.
"""

    user_content = f"### USER PROMPT:\n{prompt}\n\n### CRITERIA TO CHECK:\n{criteria}\n\n### GENERATED CODE:\n{generated_code}"

    async with SEMAPHORE:
        try:
            response = await client.chat.completions.create(
                model="gpt-4o",
                response_format={ "type": "json_object" },
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content}
                ],
                temperature=0.1
            )
            result = json.loads(response.choices[0].message.content)
            return result.get("score", 1), result.get("reason", "Không thể parse lý do")
        except Exception as e:
            return 1, f"Lỗi gọi LLM: {str(e)}"

In [ ]:
async def process_record(item, base_code, lora_code):
    task_id = item["id"]
    prompt = item["messages"][0]["content"]
    criteria = item.get("evaluation_metrics", {}).get("judge_criteria", "Code must be correct and robust.")
    category = item.get("category", "General")

    (base_score, base_reason), (lora_score, lora_reason) = await asyncio.gather(
        llm_judge(prompt, base_code, criteria),
        llm_judge(prompt, lora_code, criteria)
    )

    if lora_score > base_score:
        status = "LoRA Win"
    elif lora_score < base_score:
        status = "LoRA Lose"
    else:
        status = "Draw"

    return {
        "Task ID": task_id,
        "Category": category,
        "Base Score": base_score,
        "Base Reason": base_reason,
        "LoRA Score": lora_score,
        "LoRA Reason": lora_reason,
        "Status": status
    }

In [ ]:
async def main():
    with open("data/processed/eval_dataset.json", "r", encoding="utf-8") as f:
        eval_data = json.load(f)
    with open("notebooks/results_base.json", "r", encoding="utf-8") as f:
        base_results = {b["id"]: b["generated_code"] for b in json.load(f)}
    with open("notebooks/results_lora.json", "r", encoding="utf-8") as f:
        lora_results = {b["id"]: b["generated_code"] for b in json.load(f)}

    print(f"LLM Judge scoring {len(eval_data)} tasks...")

    tasks = []
    for item in eval_data:
        base_code = base_results.get(item["id"], "")
        lora_code = lora_results.get(item["id"], "")
        tasks.append(process_record(item, base_code, lora_code))

    report_data = await tqdm_asyncio.gather(*tasks)

    df = pd.DataFrame(report_data)

    print("\n=========================================")
    print("RESULT (LLM-AS-A-JUDGE)")
    print("=========================================")
    print(f"Base Model: {df['Base Score'].mean():.2f} / 5.0")
    print(f"LoRA Model: {df['LoRA Score'].mean():.2f} / 5.0")
    print("-----------------------------------------")
    print("Versus rate:")
    print(df['Status'].value_counts().to_string())
    print("-----------------------------------------")

    print("\nScore for each Category:")
    cat_group = df.groupby('Category')[['Base Score', 'LoRA Score']].mean().round(2)
    cat_group['Diff (LoRA - Base)'] = cat_group['LoRA Score'] - cat_group['Base Score']
    print(cat_group)

    df.to_csv("llm_judge_report.csv", index=False, encoding="utf-8-sig")
    print("Saved report")

asyncio.run(main())